# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR2](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")  # .name and .description are attributes

## 2. Data Overview

Review available record sets, fields (columns), and their `@id` identifiers.

We'll inspect the record sets defined in this dataset, and for each, list its available fields and their IDs.

In [ ]:
# List all record sets by their @id with column (field) details
# All Croissant entities are referenced by their @id

print("Available record sets and their fields:\n")
record_sets = dataset.record_sets  # This is a list of CroissantRecordSet
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {getattr(rs, 'description', '-')}")
    print(f"  Fields (columns):")
    for field in rs.fields:
        print(f"    - @id: {field.id}\n      name: {field.name}\n      datatype: {field.data_type}")
    print('-' * 60)

# For further use, collect record_set @ids
record_set_ids = [rs.id for rs in record_sets]
print(f"All record set @ids: {record_set_ids}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For this dataset, there is likely one main record set with all clinical variables.
# We'll extract all record sets for demonstration.

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nExtracting records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print('No records found for this record set.')
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print("Columns available:", df.columns.tolist())
    display(df.head())

# For demonstration in next steps, pick the main record set id
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # First set
    print(f"Using {main_record_set_id} for EDA.")
else:
    print('No dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps: filter records, normalize numeric fields, and group/categorize, referencing fields by `@id`. We'll:
- Filter on a numeric field (e.g., age) if available
- Normalize that field
- Group by a categorical field (e.g., sex) if present
- All references use `@id` for fields

In [ ]:
# EDA requires us to reference fields by their @id
import numpy as np

# Choose first DataFrame loaded (main clinical table)
df = dataframes[main_record_set_id]
fields = None
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        fields = rs.fields
        break
field_id_to_col = {f.id: f.name for f in fields}

# Find a numeric field: look for age or duration field types, fallback to first float/integer
numeric_field_id = None
for f in fields:
    if (f.data_type or '').lower() in ('float','integer','number'):
        numeric_field_id = f.id
        break
if not numeric_field_id:
    # fallback: pick first column
    numeric_field_id = df.columns[0]

print(f"Numeric field selected (by @id): {numeric_field_id}")
col_numeric = field_id_to_col.get(numeric_field_id, numeric_field_id)

# Set threshold to mean
if pd.api.types.is_numeric_dtype(df[col_numeric]):
    threshold = float(df[col_numeric].mean())
else:
    threshold = 0

filtered_df = df[df[col_numeric] > threshold]
print(f"Filtered records with '{col_numeric}' (@id {numeric_field_id}) > {threshold:.2f}:")
display(filtered_df.head())

# Normalize field
filtered_df[f"{col_numeric}_normalized"] = (filtered_df[col_numeric] - filtered_df[col_numeric].mean()) / filtered_df[col_numeric].std(ddof=0)
print(f"Normalized '{col_numeric}' for filtered records:")
display(filtered_df[[col_numeric, f"{col_numeric}_normalized"]].head())

# Group by a categorical field (@id): pick next available string/categorical
group_field_id = None
for f in fields:
    # Typically 'sex', 'anatomical_site', etc. are string/categorical
    if (f.data_type or '').lower() == 'text' and f.id != numeric_field_id:
        group_field_id = f.id
        break
if group_field_id is not None and field_id_to_col[group_field_id] in filtered_df:
    group_col = field_id_to_col[group_field_id]
    grouped_df = filtered_df.groupby(group_col)[col_numeric].agg(['mean','count'])
    print(f"Grouped by '{group_col}' (@id {group_field_id}):")
    display(grouped_df.head())
else:
    print("No categorical group field found for grouping.")

## 5. Visualization

Visualize key data distributions and relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field (after normalization)
if f"{col_numeric}_normalized" in filtered_df:
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[f"{col_numeric}_normalized"], kde=True, bins=15, color='skyblue')
    plt.title(f"Distribution of normalized {col_numeric}")
    plt.xlabel(f"{col_numeric}_normalized")
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by group if available
if group_field_id is not None and group_col in filtered_df:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=filtered_df, x=group_col, y=col_numeric)
    plt.title(f"{col_numeric} by {group_col}")
    plt.xlabel(group_col)
    plt.ylabel(col_numeric)
    plt.show()
else:
    print('No categorical field available for boxplot.')

## 6. Conclusion

- We've demonstrated how to load a dataset conforming to the Croissant schema using the `mlcroissant` library, referencing all entities by their `@id`.
- We explored the structure of the "Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors" dataset.
- We extracted, filtered, normalized, grouped, and visualized key variables, with all operations tied to `@id` references for full reproducibility and schema compliance.

This notebook serves as a reproducible example for FAIR data science with Croissant-formatted datasets.